# 3 · Text-Based Audio Retrieval with CLAP

CLAP maps audio and text into a shared 512-d embedding space, enabling free-text queries over an audio collection.
This notebook retrieves sounds from the BSD10k subset by text query and analyses how prompt wording affects results.

**Questions:**
1. Does retrieval make perceptual sense?
2. How sensitive is it to specificity (`"dog"` vs. `"aggressive dog barking outside"`)?
3. Where does the model fail, and why?


## 0 · Setup

In [1]:
import os, sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from IPython.display import display, IFrame
import laion_clap, librosa

METADATA_FOLDER  = '../metadata'
EMBEDDINGS_FOLDER = '../embeddings'
SUBSET_CSV       = '../metadata/subset_metadata.csv'

def show_sound_player(sound_id):
    display(IFrame(f'https://freesound.org/embed/sound/iframe/{sound_id}/simple/medium/', width=596, height=100))


/home/frederic/Documents/UPF/semester3/advanced/block2_assignment/sound-classification-repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1 · Load CLAP embeddings and build index

In [2]:
subset_df = pd.read_csv(SUBSET_CSV)

clap_folder = os.path.join(EMBEDDINGS_FOLDER, 'clap')
sound_ids   = subset_df['sound_id'].tolist()
paths       = [os.path.join(clap_folder, f'{sid}.npy') for sid in sound_ids]

valid_mask  = np.array([os.path.exists(p) for p in paths])
print(f'CLAP embeddings found: {valid_mask.sum()} / {len(paths)}')

example = np.load(paths[valid_mask.argmax()])
X_clap  = np.zeros((len(paths), len(example)))
for i, (p, ok) in enumerate(zip(paths, valid_mask)):
    if ok:
        X_clap[i] = np.load(p)

# Build nearest-neighbour index (only over valid embeddings)
index_df = subset_df[valid_mask].reset_index(drop=True)
X_index  = X_clap[valid_mask]

nbrs = NearestNeighbors(n_neighbors=5, algorithm='ball_tree').fit(X_index)
print('Index ready.')


CLAP embeddings found: 7205 / 7205
Index ready.


## 2 · Load CLAP text encoder

In [3]:
print('Loading CLAP model…')
clap_model = laion_clap.CLAP_Module(enable_fusion=True)
clap_model.load_ckpt(model_id=3)
print('Ready.')

def text_embed(query: str) -> np.ndarray:
    return clap_model.get_text_embedding([query])[0, :]

def retrieve(query: str, k: int = 5):
    """Return top-k results for a free-text query."""
    qvec = text_embed(query)
    dists, idxs = nbrs.kneighbors([qvec], n_neighbors=k)
    results = []
    for dist, idx in zip(dists[0], idxs[0]):
        row = index_df.iloc[idx]
        results.append({'sound_id': row['sound_id'], 'title': row['title'],
                        'class': row['class'], 'top_class': row['class_name'], 'distance': float(dist)})
    return results

def print_results(query, results):
    print(f'\nQuery: "{query}"')
    print(f'  {'Rank':>4}  {'Dist':>6}  {'Class'}  {'Top Class':<28}  Title')
    print('  ' + '-' * 70)
    for i, r in enumerate(results, 1):
        print(f'  {i:>4}  {r["distance"]:>6.3f}  {r["class"]}  {r["top_class"]:<28}  {r["title"]}')


Loading CLAP model…


/home/frederic/Documents/UPF/semester3/advanced/block2_assignment/sound-classification-repo/.venv/lib/python3.12/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4382.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 14007.80it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not 

Load our best checkpoint in the paper.
The checkpoint is already downloaded
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.patch_embed.mel_conv2d.weight 	 Loaded
audio_branch.patch_embed.mel_conv2d.bias 	 Loaded
audio_branch.patch_embed.fusion_model.local_att.0.weight 	 Loaded
audio_branch.patch_embed.fusion_model.local_att.0.bias 	 Loaded
audio_branch.patch_embed.fusion_model.local_att.1.weight 	 Loaded
audio_branch.patch_embed.fusion_model.local_att.1.bias 	 Loaded
audio_branch.patch_embed.fusion_model.local_att.3.weight 	 Loaded
audio_branc

## 3 · Basic queries

In [4]:
QUERIES = [
    'rain falling on leaves',
    'dog barking',
    'crowd applauding',
    'electric guitar riff',
    'car engine starting',
    'baby crying',
    'bird singing in a forest',
]

for q in QUERIES:
    results = retrieve(q)
    print_results(q, results)
    for r in results:
        show_sound_player(r['sound_id'])
    print()



Query: "rain falling on leaves"
  Rank    Dist  Class  Top Class                     Title
  ----------------------------------------------------------------------
     1   1.025  fx-n  Sound effects                 Wind in a tree, close to the foliage. (Anatolia-Turkey)
     2   1.033  fx-n  Sound effects                 2013-03-28 rain in the rainforest.wav
     3   1.063  fx-n  Sound effects                 jp_rainloop11.flac
     4   1.077  fx-n  Sound effects                 Tent - int - rain on canvas.wav
     5   1.080  fx-n  Sound effects                 rain_session_41sec_2007.wav




Query: "dog barking"
  Rank    Dist  Class  Top Class                     Title
  ----------------------------------------------------------------------
     1   0.937  fx-a  Sound effects                 dog 2.wav
     2   0.950  fx-a  Sound effects                 Dog Barking
     3   1.020  fx-a  Sound effects                 Dog Bark
     4   1.024  fx-a  Sound effects                 20091211.barking.stairs.wav
     5   1.030  ss-n  Soundscapes                   Dog Barking.wav




Query: "crowd applauding"
  Rank    Dist  Class  Top Class                     Title
  ----------------------------------------------------------------------
     1   1.053  ss-u  Soundscapes                   Crowd_cheering_football_01.wav
     2   1.096  ss-u  Soundscapes                   Fenway Ambience 1.m4a
     3   1.117  ss-u  Soundscapes                   stazione BO 01.mp3
     4   1.120  fx-n  Sound effects                 fdv_forezan_03.wav
     5   1.121  sp-c  Speech                        sing along, large crowd, big party.wav




Query: "electric guitar riff"
  Rank    Dist  Class  Top Class                     Title
  ----------------------------------------------------------------------
     1   1.070  m-si  Music                         Breaks Driver Guitar 01.mp3
     2   1.076  m-si  Music                         Funky Electric Guitar Sample 4 - 90 BPM
     3   1.102  m-si  Music                         LOOP 12.mp3
     4   1.107  ss-u  Soundscapes                   washington_mono_bustrumpetambience.mp3
     5   1.116  m-si  Music                         Guitare3 24.wav




Query: "car engine starting"
  Rank    Dist  Class  Top Class                     Title
  ----------------------------------------------------------------------
     1   1.008  fx-v  Sound effects                 1940 'Fodge' First Start.wav
     2   1.031  fx-v  Sound effects                 STE-014.wav
     3   1.059  fx-v  Sound effects                 0320 Engine.wav
     4   1.075  fx-o  Sound effects                 Heavy hit on tube with water, secondarily hitting a fence 
     5   1.091  fx-o  Sound effects                 Idle.wav




Query: "baby crying"
  Rank    Dist  Class  Top Class                     Title
  ----------------------------------------------------------------------
     1   1.005  ss-i  Soundscapes                   babywalmartambience.wav
     2   1.039  ss-i  Soundscapes                   busridetoairportscream.mp3
     3   1.075  fx-a  Sound effects                 20070407.slaughter.04.WAV
     4   1.107  ss-n  Soundscapes                   Ben Shewmaker -  Griffey Park Geese.mp3
     5   1.115  ss-n  Soundscapes                   Sunday 03 080609.wav




Query: "bird singing in a forest"
  Rank    Dist  Class  Top Class                     Title
  ----------------------------------------------------------------------
     1   1.070  ss-u  Soundscapes                   090403_00_neu-isenburg_soundscape_birds_cars_airplane.wav
     2   1.070  fx-v  Sound effects                 steam train.WAV
     3   1.092  ss-n  Soundscapes                   park in Hamburg.WAV
     4   1.094  ss-n  Soundscapes                   Filsoe project - pedersholm 02 2012-09-08.wav
     5   1.107  ss-n  Soundscapes                   near the beach Ho Bugt.wav


## 4 · Prompt sensitivity

In [5]:
def compare_queries(variants, k=5):
    """Run multiple phrasings, print results, report Jaccard overlap."""
    all_results = {}
    for q in variants:
        r = retrieve(q, k=k)
        all_results[q] = r
        print_results(q, r)
        for result in r:
            show_sound_player(result['sound_id'])

    # Jaccard similarity between retrieved sets
    ids = {q: set(r['sound_id'] for r in res) for q, res in all_results.items()}
    print('\n  Jaccard overlap between result sets:')
    qs = list(ids.keys())
    for i in range(len(qs)):
        for j in range(i+1, len(qs)):
            a, b = ids[qs[i]], ids[qs[j]]
            j_sim = len(a & b) / len(a | b)
            print(f'    "{qs[i][:40]}" vs "{qs[j][:40]}": {j_sim:.2f}')
    return all_results


In [6]:
# Experiment 1: specificity — rain
_ = compare_queries(['rain', 'heavy rain', 'heavy rain hitting a metal roof'])



Query: "rain"
  Rank    Dist  Class  Top Class                     Title
  ----------------------------------------------------------------------
     1   0.917  fx-n  Sound effects                 Rain pouring on Leafy surface - Sample - WAV
     2   0.953  fx-n  Sound effects                 rain_session_41sec_2007.wav
     3   0.961  fx-n  Sound effects                 rain.wav
     4   0.967  fx-n  Sound effects                 Wind in a tree, close to the foliage. (Anatolia-Turkey)
     5   0.973  fx-n  Sound effects                 2013-03-28 rain in the rainforest.wav



Query: "heavy rain"
  Rank    Dist  Class  Top Class                     Title
  ----------------------------------------------------------------------
     1   0.963  ss-u  Soundscapes                   rain1.mp3
     2   0.983  fx-n  Sound effects                 Rain.wav
     3   0.988  fx-n  Sound effects                 Rain pouring on Leafy surface - Sample - WAV
     4   0.994  fx-n  Sound effects                 jp_rainloop11.flac
     5   0.998  fx-n  Sound effects                 rain_session_41sec_2007.wav



Query: "heavy rain hitting a metal roof"
  Rank    Dist  Class  Top Class                     Title
  ----------------------------------------------------------------------
     1   0.893  fx-n  Sound effects                 Rain Gutter-SPB-040.wav
     2   0.894  fx-n  Sound effects                 Rain_00029.wav
     3   0.900  fx-n  Sound effects                 rain from inside.WAV
     4   0.914  fx-n  Sound effects                 rain.WAV
     5   0.916  fx-n  Sound effects                 Rain Gutter-SPB-041.wav



  Jaccard overlap between result sets:
    "rain" vs "heavy rain": 0.25
    "rain" vs "heavy rain hitting a metal roof": 0.00
    "heavy rain" vs "heavy rain hitting a metal roof": 0.00


In [7]:
# Experiment 2: musical instrument
_ = compare_queries(['guitar', 'acoustic guitar', 'single acoustic guitar chord strummed slowly'])



Query: "guitar"
  Rank    Dist  Class  Top Class                     Title
  ----------------------------------------------------------------------
     1   1.028  m-si  Music                         Breaks Driver Guitar 01.mp3
     2   1.039  m-si  Music                         Funky Electric Guitar Sample 4 - 90 BPM
     3   1.085  m-si  Music                         LOOP 12.mp3
     4   1.085  m-si  Music                         Rhythmguitar Dmaj P107
     5   1.101  m-si  Music                         G-G 80's .mp3



Query: "acoustic guitar"
  Rank    Dist  Class  Top Class                     Title
  ----------------------------------------------------------------------
     1   0.990  is-s  Instrument samples            chord_Eflatm6.wav
     2   1.040  m-si  Music                         Rhythmguitar Dmaj P107
     3   1.055  m-si  Music                         G-G 80's .mp3
     4   1.064  is-s  Instrument samples            chord_Eflatm7.wav
     5   1.072  is-s  Instrument samples            yamaha_C7.wav



Query: "single acoustic guitar chord strummed slowly"
  Rank    Dist  Class  Top Class                     Title
  ----------------------------------------------------------------------
     1   0.842  is-s  Instrument samples            chord_Eflatm6.wav
     2   0.868  is-s  Instrument samples            Flamenco Finale 1.wav
     3   0.881  is-s  Instrument samples            chord_Gaug.wav
     4   0.881  is-s  Instrument samples            chord_Eflatm7.wav
     5   0.885  is-s  Instrument samples            chord_Fsharp7.wav



  Jaccard overlap between result sets:
    "guitar" vs "acoustic guitar": 0.25
    "guitar" vs "single acoustic guitar chord strummed sl": 0.00
    "acoustic guitar" vs "single acoustic guitar chord strummed sl": 0.25


In [8]:
# Experiment 3: human voice
_ = compare_queries(['voice', 'speech', 'person speaking quietly indoors'])



Query: "voice"
  Rank    Dist  Class  Top Class                     Title
  ----------------------------------------------------------------------
     1   1.027  sp-s  Speech                        Male_Scream_11.wav
     2   1.064  sp-s  Speech                        Male Voice - Welcome
     3   1.069  sp-s  Speech                        thereisnogodwhisper.mp3
     4   1.069  sp-s  Speech                        quick overe here edit.wav
     5   1.075  sp-s  Speech                        derukugiwautareru(Connum)_1.wav



Query: "speech"
  Rank    Dist  Class  Top Class                     Title
  ----------------------------------------------------------------------
     1   1.078  sp-s  Speech                        epanody--come_one_you_said_you_would.wav
     2   1.096  sp-s  Speech                        fingerprint authorization accepted.wav
     3   1.105  sp-s  Speech                        how_do_i_find.wav
     4   1.105  sp-s  Speech                        4th and king.wav
     5   1.110  fx-n  Sound effects                 Gurgle.wav



Query: "person speaking quietly indoors"
  Rank    Dist  Class  Top Class                     Title
  ----------------------------------------------------------------------
     1   1.088  ss-i  Soundscapes                   stepsmuseum3.wav
     2   1.124  ss-u  Soundscapes                   City Late Summer Night NL 130726_02.wav
     3   1.128  ss-i  Soundscapes                   Cologne cathedral.WAV
     4   1.139  ss-n  Soundscapes                   STE-027-edit.wav
     5   1.158  fx-n  Sound effects                 Gurgle.wav



  Jaccard overlap between result sets:
    "voice" vs "speech": 0.00
    "voice" vs "person speaking quietly indoors": 0.00
    "speech" vs "person speaking quietly indoors": 0.11


### Observations — prompt sensitivity

> **TODO:** Discuss the Jaccard overlaps and any surprising changes in the result sets.

Hypotheses to address:
- Does more specificity consistently improve precision, or can it hurt?
- Can a class-label word (e.g. `"soundscape"`) be used directly as a query?
- What happens with negation-like prompts (`"not music"`, `"silence"`)?


## 5 · Precision@k evaluation

In [9]:
# Measure how often the top-k results match the expected class
LABELED_QUERIES = [
    ('rain falling on leaves',    'fx-n'),
    ('dog barking',               'fx-a'),
    ('crowd applauding',          'ss-u'),
    ('electric guitar riff',      'm-si'),
    ('car engine starting',       'fx-v'),
    ('baby crying',               'sp-s'),
    ('bird singing in a forest',  'ss-n'),
]
# NOTE: update the expected class names to match your actual class_name values

print(f'  {'Query':<45}  {'Expected class':<26}  P@5')
print('  ' + '-' * 80)
for query, expected_cls in LABELED_QUERIES:
    results = retrieve(query, k=5)
    correct = sum(1 for r in results if expected_cls.lower() in r['class'].lower())
    print(f'  {query:<45}  {expected_cls:<26}  {correct/5:.2f}')


  Query                                          Expected class              P@5
  --------------------------------------------------------------------------------
  rain falling on leaves                         fx-n                        1.00
  dog barking                                    fx-a                        0.80
  crowd applauding                               ss-u                        0.60
  electric guitar riff                           m-si                        0.80
  car engine starting                            fx-v                        0.60
  baby crying                                    sp-s                        0.00
  bird singing in a forest                       ss-n                        0.60
